In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Activity/'
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Videos will be saved to: {SAVE_DIR}")

Mounted at /content/drive
Videos will be saved to: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Activity/


In [ ]:
# ============================================================
# CELL 2 — Install dependencies
# ============================================================
!pip install -q yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# CELL 3 — Download ActivityNet v1.3 annotations (from GitHub)
# ============================================================
import urllib.request, json

# ✅ Official GitHub mirror — EC2 server is permanently down
ANNOT_URL = "https://raw.githubusercontent.com/activitynet/ActivityNet/master/Evaluation/data/activity_net.v1-3.min.json"
ANNOT_PATH = "/content/activity_net.v1-3.min.json"

print("Downloading annotation file from GitHub...")
urllib.request.urlretrieve(ANNOT_URL, ANNOT_PATH)

with open(ANNOT_PATH) as f:
    data = json.load(f)

print(f"✓ Loaded {len(data['database'])} video entries.")

✓ Loaded 19994 video entries.


In [ ]:
# ============================================================
# CELL 4 — Select 100 videos: 1 per distinct activity class
# ============================================================
from collections import defaultdict

label_to_videos = defaultdict(list)

for vid_id, info in data['database'].items():
    subset = info.get('subset', '')
    annotations = info.get('annotations', [])
    if subset in ('training', 'validation') and annotations:
        label = annotations[0]['label']
        label_to_videos[label].append(vid_id)

print(f"Found {len(label_to_videos)} distinct activity classes.")

selected = []
for label, vids in sorted(label_to_videos.items()):
    selected.append((label, vids[0]))
    if len(selected) == 100:
        break

print(f"\nSelected {len(selected)} videos across {len(selected)} distinct themes.")
print("\nSample classes:")
for label, vid in selected[:10]:
    print(f"  [{label}] → https://www.youtube.com/watch?v={vid}")

Found 200 distinct activity classes.

Selected 100 videos across 100 distinct themes.

Sample classes:
  [Applying sunscreen] → https://www.youtube.com/watch?v=QJm_B5Hx4DI
  [Archery] → https://www.youtube.com/watch?v=UCOn2HkJJt8
  [Arm wrestling] → https://www.youtube.com/watch?v=A1EflBqBv14
  [Assembling bicycle] → https://www.youtube.com/watch?v=BfnM0eyjB5Q
  [BMX] → https://www.youtube.com/watch?v=6uhLrPgbpUA
  [Baking cookies] → https://www.youtube.com/watch?v=IGcsVPa34Hc
  [Ballet] → https://www.youtube.com/watch?v=c_NlYvL96y0
  [Bathing dog] → https://www.youtube.com/watch?v=V90aT-d_FKo
  [Baton twirling] → https://www.youtube.com/watch?v=dAa10hlgxCY
  [Beach soccer] → https://www.youtube.com/watch?v=IoGpS8NQklE


In [ ]:
# ============================================================
# CELL 5 — Download videos to Google Drive
# ============================================================
import subprocess, time

failed = []
success = 0

for i, (label, vid_id) in enumerate(selected, 1):
    url = f"https://www.youtube.com/watch?v={vid_id}"
    safe_label = label.replace('/', '-').replace(' ', '_')
    output_template = os.path.join(SAVE_DIR, f"{safe_label}__{vid_id}.%(ext)s")

    print(f"\n[{i:03d}/100] {label} → {vid_id}")

    result = subprocess.run([
        "yt-dlp",
        "--no-playlist",
        "-f", "bestvideo[height<=480]+bestaudio/best[height<=480]",
        "--merge-output-format", "mp4",
        "--socket-timeout", "30",
        "--retries", "3",
        "-o", output_template,
        url
    ], capture_output=True, text=True)

    if result.returncode == 0:
        success += 1
        print(f"  ✓ Downloaded successfully")
    else:
        failed.append((label, vid_id))
        err_lines = [l for l in result.stderr.strip().splitlines() if l]
        print(f"  ✗ Failed: {err_lines[-1] if err_lines else 'unknown error'}")

    time.sleep(1)

print(f"\n{'='*50}")
print(f"Done! {success}/100 videos downloaded successfully.")
if failed:
    print(f"\nFailed ({len(failed)}):")
    for label, vid in failed:
        print(f"  [{label}] {vid}")


[001/100] Applying sunscreen → QJm_B5Hx4DI
  ✗ Failed: ERROR: [youtube] QJm_B5Hx4DI: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

[002/100] Archery → UCOn2HkJJt8
  ✓ Downloaded successfully

[003/100] Arm wrestling → A1EflBqBv14
  ✓ Downloaded successfully

[004/100] Assembling bicycle → BfnM0eyjB5Q
  ✓ Downloaded successfully

[005/100] BMX → 6uhLrPgbpUA
  ✓ Downloaded successfully

[006/100] Baking cookies → IGcsVPa34Hc
  ✓ Downloaded successfully

[007/100] Ballet → c_NlYvL96y0
  ✓ Downloaded successfully

[008/100] Bathing dog → V90aT-d_FKo
  ✓ Downloaded successfully

[009/100] Baton twirling → dAa10hlgxCY
  ✓ Downloaded successfully

[010/100] Bea

In [ ]:
# ============================================================
# CELL 6 — (Optional) Save a CSV manifest of downloaded videos
# ============================================================
import csv

manifest_path = os.path.join(SAVE_DIR, "manifest.csv")
with open(manifest_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["index", "label", "youtube_id", "url"])
    for i, (label, vid_id) in enumerate(selected, 1):
        writer.writerow([i, label, vid_id, f"https://www.youtube.com/watch?v={vid_id}"])

print(f"Manifest saved to {manifest_path}")

Manifest saved to /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Activity/manifest.csv


In [ ]:
import os
import subprocess
import glob

# --- CONFIGURATION ---
VIDEO_FOLDER = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Activity/'
FRAMES_FOLDER = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/frames/'
FPS = 2

os.makedirs(FRAMES_FOLDER, exist_ok=True)

def extract_frames(video_path, output_dir, fps=2):
    """Extract frames at given fps using ffmpeg."""
    os.makedirs(output_dir, exist_ok=True)
    output_pattern = os.path.join(output_dir, "frame_%04d.jpg")

    cmd = [
        'ffmpeg', '-i', video_path,
        '-vf', f'fps={fps}',
        '-q:v', '2',          # high jpeg quality
        output_pattern,
        '-hide_banner', '-loglevel', 'error'
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"✗ ffmpeg error for {os.path.basename(video_path)}:\n{result.stderr}")
        return False

    frame_count = len(glob.glob(os.path.join(output_dir, "*.jpg")))
    return frame_count

def process_all_videos(video_folder, frames_folder, fps=2):
    all_videos = glob.glob(os.path.join(video_folder, "*.mp4"))

    if not all_videos:
        print(f"No .mp4 files found in {video_folder}")
        return

    print(f"Found {len(all_videos)} videos to process.\n")

    for video_path in all_videos:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_dir = os.path.join(frames_folder, video_name)

        # Skip if already extracted
        if os.path.exists(output_dir) and len(glob.glob(os.path.join(output_dir, "*.jpg"))) > 0:
            existing = len(glob.glob(os.path.join(output_dir, "*.jpg")))
            print(f"Already extracted ({existing} frames), skipping: {video_name}")
            continue

        print(f"Extracting: {video_name}")
        frame_count = extract_frames(video_path, output_dir, fps)

        if frame_count:
            duration_approx = frame_count / fps
            print(f"✓ {video_name}: {frame_count} frames (~{duration_approx:.0f}s at {fps}fps)")
        else:
            print(f"✗ Failed: {video_name}")

    print("\nAll done.")
    print(f"Frames saved to: {frames_folder}")

# --- MAIN ---
process_all_videos(VIDEO_FOLDER, FRAMES_FOLDER, fps=FPS)

Found 77 videos to process.

Extracting: Archery__UCOn2HkJJt8
✓ Archery__UCOn2HkJJt8: 370 frames (~185s at 2fps)
Extracting: Arm_wrestling__A1EflBqBv14
✓ Arm_wrestling__A1EflBqBv14: 194 frames (~97s at 2fps)
Extracting: Assembling_bicycle__BfnM0eyjB5Q
✓ Assembling_bicycle__BfnM0eyjB5Q: 371 frames (~186s at 2fps)
Extracting: BMX__6uhLrPgbpUA
✓ BMX__6uhLrPgbpUA: 92 frames (~46s at 2fps)
Extracting: Baking_cookies__IGcsVPa34Hc
✓ Baking_cookies__IGcsVPa34Hc: 429 frames (~214s at 2fps)
Extracting: Ballet__c_NlYvL96y0
✓ Ballet__c_NlYvL96y0: 250 frames (~125s at 2fps)
Extracting: Bathing_dog__V90aT-d_FKo
✓ Bathing_dog__V90aT-d_FKo: 156 frames (~78s at 2fps)
Extracting: Baton_twirling__dAa10hlgxCY
✓ Baton_twirling__dAa10hlgxCY: 457 frames (~228s at 2fps)
Extracting: Beach_soccer__IoGpS8NQklE
✓ Beach_soccer__IoGpS8NQklE: 464 frames (~232s at 2fps)
Extracting: Beer_pong__JDg--pjY5gg
✓ Beer_pong__JDg--pjY5gg: 252 frames (~126s at 2fps)
Extracting: Belly_dance__3K62qZ2hGyw
✓ Belly_dance__3K62qZ2hG

In [ ]:
import os
import subprocess
import glob

# --- CONFIGURATION ---
VIDEO_FOLDER = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Animals/'
FRAMES_FOLDER = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Animals/frames/'
FPS = 2

os.makedirs(FRAMES_FOLDER, exist_ok=True)

def extract_frames(video_path, output_dir, fps=2):
    """Extract frames at given fps using ffmpeg."""
    os.makedirs(output_dir, exist_ok=True)
    output_pattern = os.path.join(output_dir, "frame_%04d.jpg")

    cmd = [
        'ffmpeg', '-i', video_path,
        '-vf', f'fps={fps}',
        '-q:v', '2',          # high jpeg quality
        output_pattern,
        '-hide_banner', '-loglevel', 'error'
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"✗ ffmpeg error for {os.path.basename(video_path)}:\n{result.stderr}")
        return False

    frame_count = len(glob.glob(os.path.join(output_dir, "*.jpg")))
    return frame_count

def process_all_videos(video_folder, frames_folder, fps=2):
    all_videos = glob.glob(os.path.join(video_folder, "*.mp4"))

    if not all_videos:
        print(f"No .mp4 files found in {video_folder}")
        return

    print(f"Found {len(all_videos)} videos to process.\n")

    for video_path in all_videos:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_dir = os.path.join(frames_folder, video_name)

        # Skip if already extracted
        if os.path.exists(output_dir) and len(glob.glob(os.path.join(output_dir, "*.jpg"))) > 0:
            existing = len(glob.glob(os.path.join(output_dir, "*.jpg")))
            print(f"Already extracted ({existing} frames), skipping: {video_name}")
            continue

        print(f"Extracting: {video_name}")
        frame_count = extract_frames(video_path, output_dir, fps)

        if frame_count:
            duration_approx = frame_count / fps
            print(f"✓ {video_name}: {frame_count} frames (~{duration_approx:.0f}s at {fps}fps)")
        else:
            print(f"✗ Failed: {video_name}")

    print("\nAll done.")
    print(f"Frames saved to: {frames_folder}")

# --- MAIN ---
process_all_videos(VIDEO_FOLDER, FRAMES_FOLDER, fps=FPS)

Found 9 videos to process.

Extracting: Buffaloes
✓ Buffaloes: 155 frames (~78s at 2fps)
Extracting: Elephant_ANP
✓ Elephant_ANP: 61 frames (~30s at 2fps)
Extracting: Warthog
✓ Warthog: 57 frames (~28s at 2fps)
Extracting: Rhinoceros-
✓ Rhinoceros-: 158 frames (~79s at 2fps)
Extracting: Lions
✓ Lions: 70 frames (~35s at 2fps)
Extracting: Zebra
✓ Zebra: 58 frames (~29s at 2fps)
Extracting: NO_SOUND_GTFU_Clips- CTPH. Copyright One Health Productions.
✓ NO_SOUND_GTFU_Clips- CTPH. Copyright One Health Productions.: 25 frames (~12s at 2fps)
Extracting: Elephants. CTPHmp4. CTPHmp4. CTPHmp4
✓ Elephants. CTPHmp4. CTPHmp4. CTPHmp4: 101 frames (~50s at 2fps)
Extracting: HIPPOPOTAMUS.CTPH
✓ HIPPOPOTAMUS.CTPH: 107 frames (~54s at 2fps)

All done.
Frames saved to: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Animals/frames/


In [ ]:
import os
import subprocess
import glob

# --- CONFIGURATION ---
VIDEO_FOLDER = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/actual_videos'
FRAMES_FOLDER = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/frames/'
FPS = 2

os.makedirs(FRAMES_FOLDER, exist_ok=True)

def extract_frames(video_path, output_dir, fps=2):
    """Extract frames at given fps using ffmpeg."""
    os.makedirs(output_dir, exist_ok=True)
    output_pattern = os.path.join(output_dir, "frame_%04d.jpg")

    cmd = [
        'ffmpeg', '-i', video_path,
        '-vf', f'fps={fps}',
        '-q:v', '2',          # high jpeg quality
        output_pattern,
        '-hide_banner', '-loglevel', 'error'
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f"✗ ffmpeg error for {os.path.basename(video_path)}:\n{result.stderr}")
        return False

    frame_count = len(glob.glob(os.path.join(output_dir, "*.jpg")))
    return frame_count

def process_all_videos(video_folder, frames_folder, fps=2):
    all_videos = glob.glob(os.path.join(video_folder, "*.mp4"))

    if not all_videos:
        print(f"No .mp4 files found in {video_folder}")
        return

    print(f"Found {len(all_videos)} videos to process.\n")

    for video_path in all_videos:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_dir = os.path.join(frames_folder, video_name)

        # Skip if already extracted
        if os.path.exists(output_dir) and len(glob.glob(os.path.join(output_dir, "*.jpg"))) > 0:
            existing = len(glob.glob(os.path.join(output_dir, "*.jpg")))
            print(f"Already extracted ({existing} frames), skipping: {video_name}")
            continue

        print(f"Extracting: {video_name}")
        frame_count = extract_frames(video_path, output_dir, fps)

        if frame_count:
            duration_approx = frame_count / fps
            print(f"✓ {video_name}: {frame_count} frames (~{duration_approx:.0f}s at {fps}fps)")
        else:
            print(f"✗ Failed: {video_name}")

    print("\nAll done.")
    print(f"Frames saved to: {frames_folder}")

# --- MAIN ---
process_all_videos(VIDEO_FOLDER, FRAMES_FOLDER, fps=FPS)

Found 43 videos to process.

Extracting: with_smoke_1Z2K6lDt76M
✓ with_smoke_1Z2K6lDt76M: 620 frames (~310s at 2fps)
Extracting: volcanic_eruption_zXA9RCmZJCY
✓ volcanic_eruption_zXA9RCmZJCY: 254 frames (~127s at 2fps)
Extracting: tropical_cyclone_Bkyod1P6LO0
✓ tropical_cyclone_Bkyod1P6LO0: 243 frames (~122s at 2fps)
Extracting: thunderstorm_fAEZa3PqkFk
✓ thunderstorm_fAEZa3PqkFk: 258 frames (~129s at 2fps)
Extracting: tornado_Kj9vYIvRdmQ
✓ tornado_Kj9vYIvRdmQ: 36 frames (~18s at 2fps)
Extracting: train_accident_ElNoz_Ac2Kk
✓ train_accident_ElNoz_Ac2Kk: 386 frames (~193s at 2fps)
Extracting: under_construction_YwXnUDw5eMk
✓ under_construction_YwXnUDw5eMk: 617 frames (~308s at 2fps)
Extracting: on_fire_JUUn0iXMNAs
✓ on_fire_JUUn0iXMNAs: 175 frames (~88s at 2fps)
Extracting: wildfire_n02ouEgZeNQ
✓ wildfire_n02ouEgZeNQ: 93 frames (~46s at 2fps)
Extracting: truck_accident_VHfjMtKKGAg
✓ truck_accident_VHfjMtKKGAg: 108 frames (~54s at 2fps)
Extracting: traffic_jam_uhwobcnhzts
✓ traffic_jam_u